In [1]:
import os
import json
import sqlite3
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from gtts import gTTS
import io

In [ ]:
env_path = Path.cwd() / ".env"
if not env_path.exists():~
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path, override=True)

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set")

MODEL = "openai/gpt-4.1-mini"

client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

OpenRouter API Key exists and begins sk-or-v1


In [3]:
system_message = """
You are a helpful assistant called StackBot, an expert Tech Stack Advisor.
You help developers choose tools, frameworks, and services for their projects.
Give short, practical answers, no more than 2-3 sentences.
If asked about pricing for a specific tool, use the pricing tool available to you.
Always be accurate. If you don't know something, say so honestly.
"""

In [4]:
DB = "tool_prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS prices (
        tool TEXT PRIMARY KEY,
        price TEXT
    )
    """)
    cursor.executemany("""
    INSERT OR REPLACE INTO prices (tool, price)
    VALUES (?, ?)
    """, [
        ("supabase", "Free tier available, Pro starts at $25/month"),
        ("vercel", "Free hobby tier, Pro starts at $20/month"),
        ("react", "Free and open-source, no pricing"),
        ("firebase", "Free Spark plan, Blaze pay-as-you-go"),
        ("openai api", "Pay-as-you-go, varies by model, e.g. $0.15 per 1M tokens for gpt-4o-mini"),
    ])
    conn.commit()

print("tool_prices.db created successfully")

tool_prices.db created successfully


In [5]:
def get_tool_price(tool_name):
    print(f"TOOL CALLED: looking up price for {tool_name}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE tool = ?', (tool_name.lower(),))
        result = cursor.fetchone()
        if result:
            return f"{tool_name}: {result[0]}"
        else:
            return f"No pricing info available for {tool_name}"

In [6]:
def get_tool_price(tool_name):
    print(f"TOOL CALLED: looking up price for {tool_name}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE tool = ?', (tool_name.lower(),))
        result = cursor.fetchone()
        if result:
            return f"{tool_name}: {result[0]}"
        else:
            return f"No pricing info available for {tool_name}"

In [7]:
get_tool_price("react")

TOOL CALLED: looking up price for react


'react: Free and open-source, no pricing'

In [8]:
price_function = {
    "name": "get_tool_price",
    "description": "Get pricing information for a development tool or service, such as Supabase, Vercel, React, Firebase, or OpenAI API.",
    "parameters": {
        "type": "object",
        "properties": {
            "tool_name": {
                "type": "string",
                "description": "The name of the tool or service the user is asking about",
            },
        },
        "required": ["tool_name"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}]

In [9]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_tool_price":
            arguments = json.loads(tool_call.function.arguments)
            tool_name = arguments.get('tool_name')
            price_info = get_tool_price(tool_name)
            responses.append({
                "role": "tool",
                "content": price_info,
                "tool_call_id": tool_call.id
            })
    return responses

In [10]:
import gtts
print(gtts.__version__)

2.5.4


In [11]:
def talker(message):
    tts = gTTS(text=message, lang='en')
    buf = io.BytesIO()
    tts.write_to_fp(buf)
    buf.seek(0)
    return buf.read()

In [12]:
audio_bytes = talker("Hello, this is a test of the text to speech system")
len(audio_bytes)

33600

In [13]:
import base64
from io import BytesIO
from PIL import Image

def artist(topic):
    prompt = f"A clean, modern software architecture diagram showing {topic}, with labeled boxes and arrows, tech/dev style, on a dark background"

    image_response = client.chat.completions.create(
        model="google/gemini-2.5-flash-image",
        messages=[{"role": "user", "content": prompt}],
        modalities=["image", "text"],
    )

    message = image_response.choices[0].message
    image_url = message.images[0]["image_url"]["url"] if isinstance(message.images[0], dict) else message.images[0].image_url.url
    image_base64 = image_url.split(",", 1)[1] if image_url.startswith("data:") else image_url
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [14]:
def chat(history):
    messages = [{"role": "system", "content": system_message}] + history

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    tool_used = False

    while response.choices[0].finish_reason == "tool_calls":
        tool_used = True
        message = response.choices[0].message
        tool_responses = handle_tool_calls(message)

        messages.append(message)
        messages.extend(tool_responses)

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    reply = response.choices[0].message.content

    audio = talker(reply)

    image = None
    if tool_used:
        last_user_msg = history[-1]["content"]
        image = artist(last_user_msg)

    return reply, audio, image

In [15]:
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

def respond(history):
    reply, audio, image = chat(history)
    history = history + [{"role": "assistant", "content": reply}]
    return history, audio, image

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Ask StackBot about tools, frameworks, or pricing:")

    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    ).then(
        respond,
        inputs=chatbot,
        outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


TOOL CALLED: looking up price for Vercel
